# MedFrameQA 论文级自动 pipeline

这份 notebook 是单入口包装器，直接调用 `run_medframeqa_paper_pipeline.py`。

默认完整流程：

1. `preflight`
2. `main5_once`
3. `recover_main5_paper_eval`
4. `posthoc_validate_main5`
5. `paper_analyze_main5`
6. `core_repeats`
7. `posthoc_validate_all`
8. `paper_analyze_all`
9. `optional_budget_ablation`


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

ROOT = Path("/gluon4/xl693/evolve")
PIPELINE_SCRIPT = ROOT / "run_medframeqa_paper_pipeline.py"
OUTPUT_DIR = ROOT / "paper_analysis_output"
REPORT_PATH = OUTPUT_DIR / "pipeline_report.json"

print("ROOT:", ROOT)
print("PIPELINE_SCRIPT:", PIPELINE_SCRIPT)
print("REPORT_PATH:", REPORT_PATH)


In [ ]:
# 只跑 preflight，确认协议和服务都正常。
cmd = [sys.executable, str(PIPELINE_SCRIPT), "--stage", "preflight"]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# 完整自动流程：默认会包含可选的 budget ablation。
cmd = [sys.executable, str(PIPELINE_SCRIPT), "--stage", "full"]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# 如果 main5_once 已经跑完，但 paper_eval 中途失败，用这格只恢复当前 5 条 repeat01 结果。
cmd = [sys.executable, str(PIPELINE_SCRIPT), "--stage", "recover_main5_paper_eval"]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# 如果你不想当前就跑 100-generation budget ablation，用这格。
cmd = [sys.executable, str(PIPELINE_SCRIPT), "--stage", "full", "--skip-budget-ablation"]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# 断点恢复示例：先从 recover_main5_paper_eval 接着恢复现有 batch。
cmd = [sys.executable, str(PIPELINE_SCRIPT), "--stage", "full", "--resume-from", "recover_main5_paper_eval"]
print("Running:", " ".join(cmd))
print("如果要真正执行，把下面一行取消注释。")
# subprocess.run(cmd, check=True)


In [ ]:
if REPORT_PATH.exists():
    report = json.loads(REPORT_PATH.read_text())
    print(json.dumps(report, indent=2, ensure_ascii=False))
else:
    print("尚未发现 pipeline_report.json")
